# NeuroSim — Notebook 04: Clinical Pipeline Demo
## AUD · ADNI · Epilepsy — Synthetic Cohort Walkthrough

**What this notebook demonstrates:**

The three clinical pipelines applied to synthetic cohorts, end-to-end.
Each section is structured identically to the real-data analysis that
runs on June 9 — the only change is swapping the synthetic BOLD generator
for `neurosim.loader.BIDSLoader`.

**Pipelines:**
- `AUDPipeline` — discordant MZ twin design, reward attractor analysis
- `ADNIPipeline` — Retrogenesis Hypothesis, disease stage trajectory
- `EpilepsyPipeline` — Seizure Onset Zone identification

**On June 9:** Replace Section 1 data factories with real HCP/ADNI/TLE
data loaded via `BIDSLoader`. Everything from Section 2 onwards is unchanged.

**References:**
- Gu et al. (2015) Nature Comms — Controllability
- Srivastava et al. (2020) PLOS Comp. Biol. — Finite-horizon NCT
- Jirsa et al. (2014) Brain — Seizure dynamics

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from neurosim.physics import normalise_matrix, minimum_energy
from neurosim.connectivity import ridge_effective_connectivity
from neurosim.clinical import AUDPipeline, ADNIPipeline, EpilepsyPipeline
from neurosim.plot import (
    plot_controllability, plot_cohort_energy,
    plot_stimulation_targets, set_style, PALETTE
)

set_style()
print("NeuroSim Notebook 04: Clinical Pipeline Demo")
print(f"NumPy {np.__version__} | Pandas {pd.__version__}")

## Section 1: Synthetic Cohort Generation
### [SYNTHETIC] — Replace with BIDSLoader on June 9

Each function generates a realistic BOLD time series where the group
difference is built into the generative model, giving us known ground truth
to validate our pipeline against.

**To use real data (June 9):**
```python
from neurosim.loader import BIDSLoader
loader = BIDSLoader('/data/HCP_S1200', atlas='schaefer400')
subjects = loader.load_cohort(
    subjects=['sub-HCP001', ...],
    connectome_dir='/data/HCP_S1200/connectomes/'
)
```

In [ ]:
# ── Synthetic data parameters ─────────────────────────────────────────────
N     = 20     # regions  (swap for 400 on real data)
T_ts  = 600    # timepoints
TR_ms = 720.0  # ms

def make_connectome(seed=0):
    rng = np.random.default_rng(seed)
    SC = np.zeros((N, N))
    for i in range(N):
        SC[i, (i+1) % N] = 1.0
        SC[(i+1) % N, i] = 1.0
        if rng.random() < 0.15:
            j = rng.integers(0, N)
            SC[i, j] = rng.uniform(0.3, 0.7)
            SC[j, i] = SC[i, j]
    np.fill_diagonal(SC, 0)
    return (SC + SC.T) / 2.0

def make_bold(seed, attractor_strength=0.0, ictal_period=None):
    rng = np.random.default_rng(seed)
    SC = make_connectome(seed)
    A_gen = normalise_matrix(SC + 0.05*rng.normal(0,1,(N,N)), 0.85)

    # Add attractor: elevated connectivity in reward nodes (0-3) for AUD
    if attractor_strength > 0:
        for i in range(4):
            for j in range(4):
                A_gen[i,j] += attractor_strength * rng.uniform(0.05, 0.15)
        A_gen = normalise_matrix(A_gen, 0.85)

    X = np.zeros((N, T_ts))
    for t in range(1, T_ts):
        X[:, t] = A_gen @ X[:, t-1] + rng.normal(0, 0.3, N)

    # Add ictal burst for epilepsy
    if ictal_period is not None:
        start, end = ictal_period
        X[:, start:end] += rng.normal(0, 2.0, (N, end-start))

    X = (X - X.mean(1,keepdims=True)) / (X.std(1,keepdims=True)+1e-8)
    return X, SC

print("Synthetic data factory ready.")
print(f"Parameters: N={N} regions, T={T_ts} TRs, TR={TR_ms}ms")

## Section 2: AUD Pipeline — Reward Attractor Analysis

**Hypothesis:** AUD-affected twins require more energy to exit the craving
state than their genetically matched healthy co-twins.

**Design:**
- 10 AUD subjects (reward network attractor built into generative model)
- 10 Control subjects (no attractor)
- 5 matched pairs (twin design)

**Primary metric:** ΔE* = E*(AUD) - E*(Control) for craving → rest transition

In [ ]:
print("Running AUD Pipeline...")
print("=" * 50)

# Generate synthetic cohort
# AUD: attractor_strength=0.15 (elevated reward connectivity)
# Control: attractor_strength=0.0

aud_subjects = []
for i in range(10):
    group  = 'AUD' if i < 5 else 'Control'
    astr   = 0.15 if group == 'AUD' else 0.0
    X, SC  = make_bold(seed=i, attractor_strength=astr)
    aud_subjects.append({
        'X': X, 'SC': SC,
        'subject_id': f'sub-AUD{i:02d}',
        'group': group,
        'metadata': {}
    })

# Run pipeline
pipe_aud = AUDPipeline(
    T=10,
    reward_indices=list(range(4)),   # regions 0-3 = reward network
    craving_percentile=75.0,
)
cohort_aud = pipe_aud.run_cohort(aud_subjects, verbose=False)

# Summary
df_aud = cohort_aud.summary()
print(f"Cohort: {cohort_aud.n_subjects} subjects, groups={cohort_aud.groups}")
print()
print(df_aud[['subject_id','group','E_craving_to_rest','E_rest_to_cognitive']].to_string(index=False))

In [ ]:
# Twin discordance analysis
pair_ids = [(f'sub-AUD{i:02d}', f'sub-AUD{i+5:02d}') for i in range(5)]
twin_df  = pipe_aud.twin_discordance_analysis(
    cohort_aud, pair_ids, transition='craving_to_rest'
)
print("Twin Discordance Analysis (craving → rest):")
print(twin_df.to_string(index=False))
print()
n_elevated = (twin_df['delta_E'] > 0).sum()
print(f"Pairs with elevated AUD energy: {n_elevated}/{len(twin_df)}")
print(f"Mean delta_E: {twin_df['delta_E'].mean():.4f}")
print(f"Mean ratio:   {twin_df['ratio'].mean():.4f}x")

In [ ]:
# Visualise AUD results
stats_aud = cohort_aud.statistics['craving_to_rest']
print(f"Statistics (craving -> rest):")
for grp in ['AUD','Control']:
    s = stats_aud[grp]
    print(f"  {grp}: mean={s['mean']:.4f}, median={s['median']:.4f}, n={s['n']}")
p = stats_aud.get('p_value', np.nan)
sig = stats_aud.get('significant_05', False)
print(f"  Mann-Whitney p={p:.4f} ({'*significant*' if sig else 'ns'})")
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: Energy by group
group_e = {
    grp: cohort_aud.energy_array('craving_to_rest', group=grp)
    for grp in cohort_aud.groups
}
for i, (grp, vals) in enumerate(group_e.items()):
    col = PALETTE['red'] if grp=='AUD' else PALETTE['blue']
    axes[0].bar(i, np.nanmean(vals), color=col, alpha=0.7, width=0.5,
                yerr=np.nanstd(vals)/np.sqrt(len(vals)), capsize=5)
    jitter = np.random.default_rng(i).uniform(-0.1, 0.1, len(vals))
    axes[0].scatter(i+jitter, vals, alpha=0.6, color=col, s=6**2, zorder=3)
axes[0].set_xticks([0,1]); axes[0].set_xticklabels(['AUD','Control'])
axes[0].set_ylabel('E* (craving → rest, T=10)')
axes[0].set_title('A: Control Energy by Group(AUD vs. Healthy Controls)')
if sig:
    y_top = max(max(v) for v in group_e.values()) * 1.1
    axes[0].plot([0,1],[y_top,y_top],'k-',lw=1.5)
    axes[0].text(0.5, y_top*1.02, f'p={p:.3f}', ha='center', fontsize=9)

# Panel B: Twin delta
axes[1].bar(range(len(twin_df)), twin_df['delta_E'],
            color=[PALETTE['red'] if d>0 else PALETTE['green']
                   for d in twin_df['delta_E']],
            alpha=0.85, edgecolor='white')
axes[1].axhline(0, color=PALETTE['grey'], lw=1.5, ls='--')
axes[1].set_xlabel('Twin pair')
axes[1].set_ylabel('ΔE* (AUD - Control)')
axes[1].set_title('B: Within-Pair Energy Difference(Positive = AUD elevated)')

# Panel C: Controllability by group
ac_aud  = np.mean([s.ac for s in cohort_aud.get_group('AUD')], axis=0)
ac_ctrl = np.mean([s.ac for s in cohort_aud.get_group('Control')], axis=0)
axes[2].plot(ac_aud,  color=PALETTE['red'],  lw=2, label='AUD', alpha=0.85)
axes[2].plot(ac_ctrl, color=PALETTE['blue'], lw=2, label='Control', alpha=0.85)
axes[2].fill_between(range(N), ac_aud, ac_ctrl,
                      where=ac_aud>ac_ctrl, alpha=0.2, color=PALETTE['red'])
axes[2].set_xlabel('Region'); axes[2].set_ylabel('Average controllability')
axes[2].set_title('C: Controllability Profile(AUD vs. Control, mean across subjects)')
axes[2].legend(fontsize=9)

plt.suptitle('AUD Pipeline Results — Reward Attractor Analysis',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('aud_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: aud_results.png")

## Section 3: ADNI Pipeline — Retrogenesis Analysis

**Hypothesis:** Modal controllability of late-myelinating regions decreases
monotonically from CN → MCI → AD, mirroring the Retrogenesis sequence.

**Design:**
- 5 CN (Cognitively Normal)
- 5 MCI (Mild Cognitive Impairment)
- 5 AD (Alzheimer's Disease)

**Built-in ground truth:** AD subjects have progressively reduced connectivity
in association regions (simulating white matter degradation).

In [ ]:
print("Running ADNI Pipeline...")
print("=" * 50)

def make_adni_bold(seed, disease_stage='CN'):
    rng = np.random.default_rng(seed)
    SC = make_connectome(seed)

    # Disease effect: reduce long-range connectivity (late-myelinating)
    # More severe = more degradation
    stage_severity = {'CN': 0.0, 'MCI': 0.3, 'AD': 0.6}
    severity = stage_severity.get(disease_stage, 0.0)

    # Degrade long-range connections (nodes > N//2 from each other)
    SC_degraded = SC.copy()
    for i in range(N):
        for j in range(N):
            if abs(i-j) > N//3:
                SC_degraded[i,j] *= (1 - severity * rng.uniform(0.5, 1.0))

    A_gen = normalise_matrix(SC_degraded + 0.03*rng.normal(0,1,(N,N)), 0.80)
    X = np.zeros((N, T_ts))
    for t in range(1, T_ts):
        X[:,t] = A_gen@X[:,t-1] + rng.normal(0, 0.3+severity*0.1, N)
    X = (X-X.mean(1,keepdims=True))/(X.std(1,keepdims=True)+1e-8)
    return X, SC_degraded

adni_subjects = []
for i, stage in enumerate(['CN']*5 + ['MCI']*5 + ['AD']*5):
    X, SC = make_adni_bold(seed=i+100, disease_stage=stage)
    adni_subjects.append({
        'X': X, 'SC': SC,
        'subject_id': f'adni-{stage}-{i:02d}',
        'group': stage,
        'metadata': {}
    })

# Run pipeline
# Late-myelinating regions: nodes in the 'outer' part of ring (high index)
late_regions = list(range(N//2, N))  # proxy for association cortex
pipe_adni = ADNIPipeline(
    T=10,
    stage_order=['CN', 'MCI', 'AD'],
    retrogenesis_regions=late_regions,
)
cohort_adni = pipe_adni.run_cohort(adni_subjects, verbose=False)

df_adni = cohort_adni.summary()
print(f"Cohort: {cohort_adni.n_subjects} subjects, groups={cohort_adni.groups}")
print()
print(df_adni[['subject_id','group','mean_ac','mean_mc']].to_string(index=False))

In [ ]:
# Stage trajectory
traj_mc  = pipe_adni.stage_trajectory(cohort_adni, metric='mc')
traj_e   = pipe_adni.stage_trajectory(cohort_adni, metric='memory_encoding')

print("Modal Controllability — Stage Trajectory:")
print(traj_mc)
print("Memory Encoding Energy — Stage Trajectory:")
print(traj_e)

# Retrogenesis score per subject
retro_scores = {
    s.subject_id: pipe_adni.retrogenesis_score(s)
    for s in cohort_adni.subjects
}
df_retro = df_adni[['subject_id','group']].copy()
df_retro['retrogenesis_score'] = df_retro['subject_id'].map(retro_scores)
print("Retrogresis Score by Stage:")
print(df_retro.groupby('group')['retrogenesis_score'].agg(['mean','std']).round(4))

In [ ]:
# Visualise ADNI results
stages = ['CN', 'MCI', 'AD']
stage_colors = [PALETTE['green'], PALETTE['orange'], PALETTE['red']]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: Modal controllability trajectory
if not traj_mc.empty:
    vals = [traj_mc.loc[s,'mean'] if s in traj_mc.index else np.nan for s in stages]
    errs = [traj_mc.loc[s,'sem']  if s in traj_mc.index else 0 for s in stages]
    axes[0].errorbar(stages, vals, yerr=errs,
                     color=PALETTE['blue'], lw=2.5, marker='o', ms=8,
                     capsize=5, label='Mean MC')
    axes[0].set_xlabel('Disease Stage')
    axes[0].set_ylabel('Mean Modal Controllability')
    axes[0].set_title('A: Modal Controllabilityacross Disease Stages')

# Panel B: Memory encoding energy trajectory
if not traj_e.empty:
    vals_e = [traj_e.loc[s,'mean'] if s in traj_e.index else np.nan for s in stages]
    errs_e = [traj_e.loc[s,'sem']  if s in traj_e.index else 0 for s in stages]
    axes[1].errorbar(stages, vals_e, yerr=errs_e,
                     color=PALETTE['red'], lw=2.5, marker='s', ms=8,
                     capsize=5, label='Memory encoding E*')
    axes[1].set_xlabel('Disease Stage')
    axes[1].set_ylabel('E* (memory encoding, T=10)')
    axes[1].set_title('B: Memory Encoding Energyacross Disease Stages')

# Panel C: Retrogenesis score by stage
for j, stage in enumerate(stages):
    group_subj = cohort_adni.get_group(stage)
    scores = [retro_scores[s.subject_id] for s in group_subj]
    jitter = np.random.default_rng(j).uniform(-0.1, 0.1, len(scores))
    axes[2].scatter(j + jitter, scores, color=stage_colors[j], s=8**2,
                    alpha=0.8, zorder=3, label=stage)
    axes[2].errorbar(j, np.mean(scores), yerr=np.std(scores)/np.sqrt(len(scores)),
                     color=stage_colors[j], capsize=5, lw=2.5, ms=10,
                     marker='D', zorder=5)

axes[2].set_xticks(range(3)); axes[2].set_xticklabels(stages)
axes[2].set_xlabel('Disease Stage')
axes[2].set_ylabel('Retrogenesis Score(late-myelinating MC / early-myelinating MC)')
axes[2].set_title('C: Retrogenesis Score(declining = supports hypothesis)')
axes[2].legend(fontsize=9)

plt.suptitle('ADNI Pipeline Results — Retrogenesis Hypothesis',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('adni_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: adni_results.png")

## Section 4: Epilepsy Pipeline — Seizure Onset Zone Identification

**Hypothesis:** The node with minimum single-input control energy to drive
the network from the interictal to the ictal state is the Seizure Onset Zone.

**Design:**
- 6 TLE patients (ictal bursts embedded in BOLD at known timepoints)
- 4 Controls (no ictal activity)

**Ground truth:** The SOZ is nodes 0–3 (hardwired into the generative model).

In [ ]:
print("Running Epilepsy Pipeline...")
print("=" * 50)

def make_epilepsy_bold(seed, is_tle=False):
    rng = np.random.default_rng(seed)
    SC = make_connectome(seed)

    # TLE: increased connectivity in SOZ region (nodes 0-3)
    if is_tle:
        for i in range(4):
            for j in range(4):
                SC[i,j] += rng.uniform(0.1, 0.3)
        SC = (SC + SC.T) / 2
        np.fill_diagonal(SC, 0)

    A_gen = normalise_matrix(SC + 0.05*rng.normal(0,1,(N,N)), 0.85)
    X = np.zeros((N, T_ts))
    for t in range(1, T_ts):
        X[:,t] = A_gen@X[:,t-1] + rng.normal(0, 0.3, N)

    # Ictal burst at known time window
    ictal_start = 400
    ictal_end   = 450
    ictal_idx   = None
    if is_tle:
        X[:4, ictal_start:ictal_end] += rng.normal(0, 2.5, (4, ictal_end-ictal_start))
        ictal_idx = list(range(ictal_start, ictal_end))

    X = (X-X.mean(1,keepdims=True))/(X.std(1,keepdims=True)+1e-8)
    return X, SC, ictal_idx

epi_subjects = []
for i in range(10):
    is_tle = i < 6
    X, SC, ictal_idx = make_epilepsy_bold(seed=i+200, is_tle=is_tle)
    epi_subjects.append({
        'X': X, 'SC': SC,
        'subject_id': f'epi-{"TLE" if is_tle else "Ctrl"}-{i:02d}',
        'group': 'TLE' if is_tle else 'Control',
        'metadata': {'ictal_indices': ictal_idx}
    })

pipe_epi = EpilepsyPipeline(
    T=10,
    preictal_window=5,
    compute_node_energies=True,
)
cohort_epi = pipe_epi.run_cohort(epi_subjects, verbose=False)
print(f"Cohort: {cohort_epi.n_subjects} subjects, groups={cohort_epi.groups}")

In [ ]:
# SOZ identification for each TLE patient
print('Seizure Onset Zone Identification:')
print('-' * 45)
true_soz = list(range(4))

soz_correct = 0
tle_subjects = cohort_epi.get_group('TLE')

for s in tle_subjects:
    soz = pipe_epi.identify_soz(s, top_k=5)
    primary = soz['primary_soz']
    top5    = soz['soz_candidates']
    correct_in_top5 = any(n in true_soz for n in top5)
    correct_primary = primary in true_soz
    if correct_in_top5:
        soz_correct += 1
    label_p = 'CORRECT' if correct_primary else 'wrong'
    label_t = 'hit' if correct_in_top5 else 'miss'
    print(f'  {s.subject_id}: Primary SOZ=Node {primary} ({label_p}), Top-5={top5} ({label_t})')

accuracy = soz_correct / len(tle_subjects) * 100
print(f'Top-5 SOZ accuracy: {soz_correct}/{len(tle_subjects)} = {accuracy:.0f}%')

barrier_df = pipe_epi.energy_barrier_analysis(cohort_epi)
print('Energy Barrier Table (interictal -> ictal):')
print(barrier_df)

In [ ]:
# Visualise Epilepsy results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: Mean per-node energy across TLE subjects
mean_node_e = np.nanmean(
    [s.node_energies for s in tle_subjects
     if s.node_energies is not None], axis=0
)
colors_soz = [PALETTE['red'] if i in true_soz else PALETTE['blue']
              for i in range(N)]
axes[0].bar(range(N), mean_node_e, color=colors_soz, alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Region (stimulation site)')
axes[0].set_ylabel('Mean E* across TLE patients')
axes[0].set_title('A: Per-Node Stimulation Energy(red = true SOZ nodes 0-3)')
axes[0].axvline(3.5, color=PALETTE['grey'], ls='--', lw=1.5,
                label='SOZ boundary')
axes[0].legend(fontsize=9)

# Panel B: TLE vs Control energy barrier
ctrl_e = cohort_epi.energy_array('interictal_to_ictal', group='Control')
tle_e  = cohort_epi.energy_array('interictal_to_ictal', group='TLE')
for i, (grp, vals, col) in enumerate([
    ('TLE',     tle_e,  PALETTE['red']),
    ('Control', ctrl_e, PALETTE['blue'])
]):
    finite = vals[np.isfinite(vals)]
    axes[1].bar(i, np.nanmean(finite), color=col, alpha=0.7, width=0.5,
                yerr=np.nanstd(finite)/np.sqrt(len(finite)), capsize=5)
    jitter = np.random.default_rng(i+99).uniform(-0.1,0.1,len(finite))
    axes[1].scatter(i+jitter, finite, alpha=0.7, color=col, s=7**2, zorder=3)
axes[1].set_xticks([0,1]); axes[1].set_xticklabels(['TLE','Control'])
axes[1].set_ylabel('E* (interictal → ictal, T=10)')
axes[1].set_title('B: Ictal Energy Barrier(lower = more seizure-prone)')

# Panel C: Controllability — TLE vs Control
ac_tle  = np.nanmean([s.ac for s in tle_subjects], axis=0)
ac_ctrl = np.nanmean([s.ac for s in cohort_epi.get_group('Control')], axis=0)
axes[2].plot(ac_tle,  color=PALETTE['red'],  lw=2, label='TLE', alpha=0.85)
axes[2].plot(ac_ctrl, color=PALETTE['blue'], lw=2, label='Control', alpha=0.85)
axes[2].axvspan(0, 3.5, alpha=0.08, color=PALETTE['red'], label='SOZ region')
axes[2].set_xlabel('Region'); axes[2].set_ylabel('Average controllability')
axes[2].set_title('C: Controllability Profile(TLE vs Control, shaded = SOZ)')
axes[2].legend(fontsize=9)

plt.suptitle('Epilepsy Pipeline Results — Seizure Onset Zone Identification',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('epilepsy_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: epilepsy_results.png")

## Section 5: Cross-Pipeline Summary

A consolidated summary of all three clinical analyses,
structured as it will appear in the methods paper results section.

In [ ]:
print("=" * 60)
print("NEUROSIM CLINICAL PIPELINE — SYNTHETIC COHORT SUMMARY")
print("=" * 60)

print("1. AUD PIPELINE")
print("-" * 40)
for grp in ['AUD', 'Control']:
    e = cohort_aud.energy_array('craving_to_rest', group=grp)
    print(f"   {grp:8s}: E* = {np.nanmean(e):.4f} ± {np.nanstd(e):.4f} (n={len(e)})")
p_aud = cohort_aud.statistics['craving_to_rest'].get('p_value', np.nan)
print(f"   Mann-Whitney p = {p_aud:.4f}")
print(f"   Twin delta_E mean = {twin_df['delta_E'].mean():.4f}")

print("2. ADNI PIPELINE (Retrogenesis)")
print("-" * 40)
for stage in ['CN', 'MCI', 'AD']:
    subj = cohort_adni.get_group(stage)
    scores = [retro_scores[s.subject_id] for s in subj]
    print(f"   {stage:5s}: Retrogenesis score = {np.mean(scores):.4f} ± {np.std(scores):.4f}")

print("3. EPILEPSY PIPELINE (SOZ Identification)")
print("-" * 40)
print(f"   Top-5 SOZ accuracy: {accuracy:.0f}%  (true SOZ = nodes 0-3)")
print(f"   TLE energy barrier:     {np.nanmean(tle_e):.4f}")
print(f"   Control energy barrier: {np.nanmean(ctrl_e):.4f}")
print(f"   Ratio (TLE/Control):    {np.nanmean(tle_e)/np.nanmean(ctrl_e):.2f}x")

print()
print("On June 9: replace Section 1 data factory with BIDSLoader.")
print("All pipeline code from Section 2 onwards runs unchanged.")